In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_dir = project_root / "data" / "processed"

issuedupes = pd.read_csv(
    processed_dir / "issuedupes_clean.csv"
)

eclipse = pd.read_csv(
    processed_dir / "eclipse_issues_clean.csv"
)

eclipse_pairs = pd.read_csv(
    processed_dir / "eclipse_duplicate_pairs.csv"
)

print("IssueDupesLite pairs:", len(issuedupes))
print("Eclipse issues:", len(eclipse))
print("Eclipse pairs:", len(eclipse_pairs))

IssueDupesLite pairs: 9957
Eclipse issues: 85156
Eclipse pairs: 13231


In [2]:
primary_issues = issuedupes[
    ["primary_id", "primary_text", "primary_issue_created_at"]
].rename(columns={
    "primary_id": "issue_id",
    "primary_text": "text",
    "primary_issue_created_at": "created_at"
})

duplicate_issues = issuedupes[
    ["duplicate_id", "duplicate_text", "duplicate_issue_created_at"]
].rename(columns={
    "duplicate_id": "issue_id",
    "duplicate_text": "text",
    "duplicate_issue_created_at": "created_at"
})

issuedupes_issues = pd.concat(
    [primary_issues, duplicate_issues],
    ignore_index=True
)

conflicts = issuedupes_issues.groupby("issue_id").agg(
    text_versions=("text", "nunique"),
    date_versions=("created_at", "nunique")
)

print(
    "Issues with conflicting text:",
    (conflicts["text_versions"] > 1).sum()
)

print(
    "Issues with conflicting timestamps:",
    (conflicts["date_versions"] > 1).sum()
)

issuedupes_issues = (
    issuedupes_issues
    .drop_duplicates(subset=["issue_id"])
    .reset_index(drop=True)
)

print("Unique issues:", len(issuedupes_issues))

Issues with conflicting text: 0
Issues with conflicting timestamps: 0
Unique issues: 19094


In [3]:
eclipse_issues = eclipse[
    ["Issue_id", "issue_text", "Created_time"]
].rename(columns={
    "Issue_id": "issue_id",
    "issue_text": "text",
    "Created_time": "created_at"
}).copy()

eclipse_issues["issue_id"] = (
    eclipse_issues["issue_id"].astype(str)
)

print("Unique Eclipse issues:",
      eclipse_issues["issue_id"].nunique())

print("Empty texts:",
      eclipse_issues["text"].fillna("").eq("").sum())

Unique Eclipse issues: 85156
Empty texts: 1


In [4]:
issuedupes_links = issuedupes[
    ["primary_id", "duplicate_id"]
].rename(columns={
    "primary_id": "source_id",
    "duplicate_id": "target_id"
}).copy()

issuedupes_links = issuedupes_links.astype(str)

print("IssueDupesLite direct links:", len(issuedupes_links))

print(
    "Unique links:",
    len(issuedupes_links.drop_duplicates())
)

print(
    "Self-referencing links:",
    (
        issuedupes_links["source_id"]
        == issuedupes_links["target_id"]
    ).sum()
)

IssueDupesLite direct links: 9957
Unique links: 9957
Self-referencing links: 0


In [5]:
eclipse_links = eclipse_pairs[
    ["Issue_id", "target_issue_id"]
].rename(columns={
    "Issue_id": "source_id",
    "target_issue_id": "target_id"
}).copy()

eclipse_links = eclipse_links.astype("int64").astype(str)

print("Eclipse direct links:", len(eclipse_links))

print(
    "Unique links:",
    len(eclipse_links.drop_duplicates())
)

print(
    "Self-referencing links:",
    (
        eclipse_links["source_id"]
        == eclipse_links["target_id"]
    ).sum()
)

Eclipse direct links: 13231
Unique links: 13231
Self-referencing links: 0


In [6]:
def build_duplicate_groups(issues, links):
    parent = {}

    def find(x):
        parent.setdefault(x, x)

        if parent[x] != x:
            parent[x] = find(parent[x])

        return parent[x]

    def union(a, b):
        root_a = find(a)
        root_b = find(b)

        if root_a != root_b:
            parent[root_b] = root_a

    for issue_id in issues["issue_id"].astype(str):
        find(issue_id)

    for source, target in links[
        ["source_id", "target_id"]
    ].itertuples(index=False, name=None):
        union(str(source), str(target))

    result = issues.copy()
    result["issue_id"] = result["issue_id"].astype(str)

    result["group_id"] = result["issue_id"].map(find)

    return result

In [7]:
issuedupes_grouped = build_duplicate_groups(
    issuedupes_issues,
    issuedupes_links
)

eclipse_grouped = build_duplicate_groups(
    eclipse_issues,
    eclipse_links
)

for name, data in [
    ("IssueDupesLite", issuedupes_grouped),
    ("Eclipse Platform", eclipse_grouped)
]:
    group_sizes = data.groupby("group_id").size()

    print(f"\n{name}")
    print("Total issues:", len(data))
    print("Total groups:", len(group_sizes))
    print(
        "Groups with multiple issues:",
        (group_sizes > 1).sum()
    )
    print("Largest group:", group_sizes.max())
    print(
        "Issues in multi-issue groups:",
        group_sizes[group_sizes > 1].sum()
    )


IssueDupesLite
Total issues: 19094
Total groups: 9157
Groups with multiple issues: 9157
Largest group: 10
Issues in multi-issue groups: 19094

Eclipse Platform
Total issues: 85156
Total groups: 71925
Groups with multiple issues: 7468
Largest group: 39
Issues in multi-issue groups: 20699


In [8]:
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    "issuedupes": (issuedupes_grouped, issuedupes_links),
    "eclipse": (eclipse_grouped, eclipse_links)
}

for name, (issues, links) in datasets.items():
    issues.to_csv(
        output_dir / f"{name}_issues_grouped.csv",
        index=False
    )

    links.to_csv(
        output_dir / f"{name}_direct_links.csv",
        index=False
    )

    print(f"Saved {name} issues and direct links.")

Saved issuedupes issues and direct links.
Saved eclipse issues and direct links.


In [9]:
for data in [issuedupes_grouped, eclipse_grouped]:
    data["issue_id"] = data["issue_id"].astype(str)

    data["created_at"] = pd.to_datetime(
        data["created_at"],
        utc=True,
        errors="coerce"
    )

    data["text"] = data["text"].fillna("").astype(str)

for links in [issuedupes_links, eclipse_links]:
    links["source_id"] = links["source_id"].astype(str)
    links["target_id"] = links["target_id"].astype(str)

In [10]:
def prepare_retrieval_benchmark(issues, links, train_fraction=0.8):
    issues = issues.copy()

    issues = issues[
        issues["text"].str.strip().ne("")
        & issues["created_at"].notna()
    ].copy()

    issues = issues.sort_values(
        ["created_at", "issue_id"]
    ).reset_index(drop=True)

    split_idx = int(len(issues) * train_fraction)
    cutoff = issues.iloc[split_idx]["created_at"]

    train = issues[
        issues["created_at"] < cutoff
    ].copy()

    test = issues[
        issues["created_at"] >= cutoff
    ].copy()

    valid_ids = set(issues["issue_id"])
    adjacency = {}

    for source, target in links[
        ["source_id", "target_id"]
    ].itertuples(index=False, name=None):

        if (
            source not in valid_ids
            or target not in valid_ids
            or source == target
        ):
            continue

        adjacency.setdefault(source, set()).add(target)
        adjacency.setdefault(target, set()).add(source)

    created_lookup = dict(
        zip(issues["issue_id"], issues["created_at"])
    )

    relevant = {}

    for query_id, query_time in zip(
        test["issue_id"], test["created_at"]
    ):
        earlier = {
            other
            for other in adjacency.get(query_id, set())
            if created_lookup[other] < query_time
        }

        if earlier:
            relevant[query_id] = earlier

    eligible_queries = test[
        test["issue_id"].isin(relevant)
    ].copy()

    return train, test, eligible_queries, relevant, issues

In [11]:
benchmarks = {}

for name, issues, links in [
    ("IssueDupesLite", issuedupes_grouped, issuedupes_links),
    ("Eclipse Platform", eclipse_grouped, eclipse_links)
]:
    train, test, eligible, relevant, all_issues = (
        prepare_retrieval_benchmark(issues, links)
    )

    benchmarks[name] = {
        "train": train,
        "test": test,
        "eligible": eligible,
        "relevant": relevant,
        "all_issues": all_issues
    }

    print(f"\n{name}")
    print("Training issues:", len(train))
    print("Test issues:", len(test))
    print("Eligible test queries:", len(eligible))
    print(
        "Percentage of test issues eligible:",
        round(100 * len(eligible) / len(test), 2),
        "%"
    )


IssueDupesLite
Training issues: 15275
Test issues: 3819
Eligible test queries: 2258
Percentage of test issues eligible: 59.13 %

Eclipse Platform
Training issues: 68124
Test issues: 17031
Eligible test queries: 2174
Percentage of test issues eligible: 12.76 %


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizers = {}
matrices = {}

for name, benchmark in benchmarks.items():

    print(f"\nProcessing {name}...")

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=2,
        max_features=100_000,
        sublinear_tf=True,
        norm="l2"
    )

    vectorizer.fit(benchmark["train"]["text"])

    all_issues = benchmark["all_issues"]

    X = vectorizer.transform(all_issues["text"])

    vectorizers[name] = vectorizer
    matrices[name] = X

    print("Documents:", X.shape[0])
    print("Vocabulary size:", X.shape[1])
    print("Nonzero features:", X.nnz)


Processing IssueDupesLite...
Documents: 19094
Vocabulary size: 100000
Nonzero features: 2389764

Processing Eclipse Platform...
Documents: 85155
Vocabulary size: 100000
Nonzero features: 11401281


In [13]:
import numpy as np

def evaluate_tfidf(
    benchmark,
    X,
    max_queries=None
):
    all_issues = benchmark["all_issues"]
    eligible = benchmark["eligible"]
    relevant = benchmark["relevant"]

    if max_queries is not None:
        eligible = eligible.head(max_queries)

    issue_ids = all_issues["issue_id"].to_numpy()
    timestamps = all_issues["created_at"].to_numpy(
        dtype="datetime64[ns]"
    )

    id_to_idx = {
        issue_id: idx
        for idx, issue_id in enumerate(issue_ids)
    }

    ranks = []
    evaluated = 0

    for query in eligible.itertuples(index=False):

        query_idx = id_to_idx[query.issue_id]

        cutoff = np.searchsorted(
            timestamps,
            np.datetime64(query.created_at.to_datetime64()),
            side="left"
        )

        if cutoff == 0:
            continue

        scores = (
            X[query_idx] @ X[:cutoff].T
        ).toarray().ravel()

        ranked_indices = np.argsort(
            -scores,
            kind="stable"
        )

        known_relevant = relevant[query.issue_id]

        relevant_indices = {
            id_to_idx[rid]
            for rid in known_relevant
            if rid in id_to_idx
            and id_to_idx[rid] < cutoff
        }

        if not relevant_indices:
            continue

        first_rank = next(
            (
                position
                for position, idx in enumerate(
                    ranked_indices, start=1
                )
                if idx in relevant_indices
            ),
            None
        )

        if first_rank is not None:
            ranks.append(first_rank)

        evaluated += 1

    ranks = np.asarray(ranks)

    results = {
        "Queries": evaluated,
        "Recall@1": np.mean(ranks <= 1),
        "Recall@5": np.mean(ranks <= 5),
        "Recall@10": np.mean(ranks <= 10),
        "MRR": np.mean(1.0 / ranks)
    }

    return results

In [14]:
pilot_results = {}

for name, benchmark in benchmarks.items():

    print(f"\nEvaluating {name}...")

    results = evaluate_tfidf(
        benchmark=benchmark,
        X=matrices[name],
        max_queries=100
    )

    pilot_results[name] = results

    print(results)


Evaluating IssueDupesLite...
{'Queries': 100, 'Recall@1': np.float64(0.49), 'Recall@5': np.float64(0.66), 'Recall@10': np.float64(0.69), 'MRR': np.float64(0.5650863547365429)}

Evaluating Eclipse Platform...
{'Queries': 100, 'Recall@1': np.float64(0.24), 'Recall@5': np.float64(0.47), 'Recall@10': np.float64(0.53), 'MRR': np.float64(0.33669830396298245)}


In [15]:
tfidf_results = {}

for name, benchmark in benchmarks.items():

    print(f"\nEvaluating {name}...")

    results = evaluate_tfidf(
        benchmark=benchmark,
        X=matrices[name]
    )

    tfidf_results[name] = results

results_df = pd.DataFrame(tfidf_results).T

display(results_df.round(4))


Evaluating IssueDupesLite...

Evaluating Eclipse Platform...


,Queries,Recall@1,Recall@5,Recall@10,MRR
IssueDupesLite,2258.0,0.4491,0.6630,0.7409,0.5478
Eclipse Platform,2174.0,0.1403,0.2783,0.3436,0.2085


In [16]:
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

results_df.index.name = "Dataset"

results_df.to_csv(
    results_dir / "tfidf_baseline_results.csv"
)

display(results_df.round(4))

,Queries,Recall@1,Recall@5,Recall@10,MRR
Dataset,,,,,
IssueDupesLite,2258.0,0.4491,0.6630,0.7409,0.5478
Eclipse Platform,2174.0,0.1403,0.2783,0.3436,0.2085
